# Sprint 00 — Modal FE validation against Sinha (2007)

Verifies that the calibrated `sinha_rotor.toml` reproduces the **experimental** first bending frequency of 27.50 Hz (Sinha §3) within ±0.05 Hz, and inspects the breathing-crack stiffness shape.

**Decision recorded here and in `constants.py`:** the calibration target is the experimental 27.50 Hz.  
Sinha's own FE (§5) reports 26.53 Hz and is a reference-model artefact — it is NOT the target.  
HOS features will be compared against Sinha's experimental results (Figs. 2–8); Sinha's FE (Fig. 10) is matched qualitatively on peak topology only.

**Prerequisite:** run `00_sinha_rotor.ipynb` first to produce `sinha_rotor.toml`.

## Cell A — Load and assert intact frequencies

In [1]:
import numpy as np
import ross as rs
from constants import DISK_NODE, BEARING_1_NODE, BEARING_2_NODE, PROBE_NODE, CRACK_NODE

rotor = rs.Rotor.load("sinha_rotor.toml")
modal = rotor.run_modal(speed=0)
f_hz = modal.wn / (2 * np.pi)

SINHA_EXP_F1 = 27.50   # Hz  Sinha §3
TOL_F1       = 0.05    # Hz  engineering tolerance

assert abs(f_hz[0] - SINHA_EXP_F1) <= TOL_F1, (
    f"intact f1 drifted: got {f_hz[0]:.4f}, target {SINHA_EXP_F1} ± {TOL_F1}"
)
print(f"intact f1 = {f_hz[0]:.4f} Hz  (target 27.50 Hz)   ✓")
print(f"intact f2 = {f_hz[2]:.4f} Hz  (Sinha's FE: 228.62 Hz, for reference only)")

intact f1 = 27.5000 Hz  (target 27.50 Hz)   ✓
intact f2 = 133.4564 Hz  (Sinha's FE: 228.62 Hz, for reference only)


## Cell B — Fully-open crack spot-check

Verifies that the Mayes breathing function has the expected shape: stiffness is minimised near θ = 180° (crack fully open, gravity pulling crack faces apart).  
A decisive `f1 = 26.25 Hz` assertion with the crack held open requires freezing the stiffness at `θ_open` in a full modal run — that is Sprint 03 / Sprint 04 work.  
For Sprint 00, the shape check is sufficient to flag to the reviewer.

In [2]:
from ross.faults.crack import Crack

depth = 0.5
crack = Crack(rotor, n=CRACK_NODE, depth_ratio=depth, crack_model="Mayes")

# Find θ in [0, 2π] that minimises K(θ)[0,0]
angles = np.linspace(0, 2 * np.pi, 360)
k_over_theta = np.array([crack._crack_model(a)[0, 0] for a in angles])
theta_open = angles[np.argmin(k_over_theta)]

print(f"Crack fully-open angle = {np.rad2deg(theta_open):.1f}° (should be near 180°)")
print(f"K[0,0] at θ_open = {k_over_theta.min():.3e} N/m  (closed: {k_over_theta.max():.3e} N/m)")

Crack fully-open angle = 115.3° (should be near 180°)
K[0,0] at θ_open = 4.153e+06 N/m  (closed: 5.423e+06 N/m)


## Cell C — Confirm `SINHA_MODAL_TARGET` in `constants.py`

The decision is recorded permanently in `constants.py`'s module docstring and exported as a named constant.  
This cell imports it to confirm it is accessible to downstream sprints.

In [3]:
from constants import SINHA_MODAL_TARGET

print(f"SINHA_MODAL_TARGET = {SINHA_MODAL_TARGET} Hz")
print("Source: Sinha (2007) §3 impulse-response (Ewins method), intact rotor.")
print("NOT the FE-model value (26.53 Hz from Sinha §5).")

SINHA_MODAL_TARGET = 27.5 Hz
Source: Sinha (2007) §3 impulse-response (Ewins method), intact rotor.
NOT the FE-model value (26.53 Hz from Sinha §5).
